# M5+ Solar Flare Forecasting


# Task 1 – Exploratory Data Analysis

## Objective

Before developing any machine learning model, it is important to understand the
characteristics of the dataset. This analysis examines the dataset structure, data
quality, target distribution, feature distributions, and relationships among variables.
The findings obtained from this analysis guide preprocessing decisions and subsequent
model development.

In [ ]:
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from IPython.display import display, Markdown

plt.style.use("ggplot")
sns.set_theme(style="whitegrid")

RANDOM_STATE = 42
ID_COLS = ["NOAA_AR", "HARPNUM"]
DATE_COL = "DATE__OBS"
TARGET = "Flare"


In [ ]:
df = pd.read_csv("training_data_M5_24.csv")
df.head()

FileNotFoundError: [Errno 2] No such file or directory: 'training_data_M5_24.csv'

## Dataset Shape and Data Types

The dataset shape provides the total number of observations and variables available for
analysis. Inspecting the data types helps distinguish numerical features from
identifier columns, which is essential for choosing what belongs in the feature set
later.

In [ ]:
print("Rows :", df.shape[0])
print("Columns :", df.shape[1])
df.info()

## Target Cleanup

Rows labelled `padding` are not usable observations for a binary Flare/No-flare model —
they appear to mark gaps or non-observational rows. They are dropped here, and the
reason (not just the fact) is worth stating explicitly: without a data dictionary
confirming what `padding` means, dropping it is an assumption, not a verified cleaning
step; flag this to a domain expert before trusting the final numbers.

In [ ]:
print(df[TARGET].value_counts(dropna=False))

df = df[df[TARGET] != "padding"].copy()

df[TARGET] = df[TARGET].map({"N": 0, "P": 1})

print(df[TARGET].value_counts())

n_pos = int((df[TARGET] == 1).sum())
n_neg = int((df[TARGET] == 0).sum())
display(Markdown(
    f"**Class imbalance:** {n_neg} negative vs {n_pos} positive rows "
    f"(~{n_neg / n_pos:.1f} : 1). Accuracy and plain ROC-AUC will be close to "
    f"meaningless at this prevalence (~{100 * n_pos / (n_pos + n_neg):.2f}% positive) — "
    f"PR-AUC, TSS and HSS are used later for that reason."
))

## Missing Values and Duplicates

Exact duplicate rows are checked first. Because rows are 12-minute snapshots of the same
active region, near-duplicate *consecutive* rows are at least as important: they are the
reason a random row-wise split leaks information between train and test (see Task 2).

In [ ]:
missing = df.isnull().sum()
missing = missing[missing > 0]
print("Missing values per column:")
print(missing)

print("\nExact duplicate rows:", df.duplicated().sum())

# Near-duplicate consecutive snapshots within the same active region: sort by region and
# time, then look at how much each numeric feature moves row-to-row.
feature_cols_preview = [c for c in df.select_dtypes(include=np.number).columns if c not in ID_COLS + [TARGET]]
df_sorted = df.sort_values(["NOAA_AR", DATE_COL])
step_diff = df_sorted.groupby("NOAA_AR")[feature_cols_preview].diff().abs()
near_dup_frac = (step_diff.fillna(0) < 1e-3).all(axis=1).mean()
display(Markdown(
    f"**Near-duplicate consecutive rows:** about {100 * near_dup_frac:.1f}% of rows change "
    f"by less than 1e-3 on every numeric feature from the previous snapshot of the same "
    f"active region. This is expected for 12-minute cadence data, and it is exactly why "
    f"random train/test splitting leaks: a near-identical row can land in both sets."
))

## Observation Timestamps

`DATE__OBS` is stored in two different formats in this file (`2023-01-03 18:36:00` and
`2023-04-08T14:48:00.000Z`). Both are parsed here so that later chronological/grouped
analysis is possible, and the resulting time range and coverage are checked for gaps.

In [ ]:
df[DATE_COL] = pd.to_datetime(df[DATE_COL], utc=True, format="mixed")

print("Date range:", df[DATE_COL].min(), "to", df[DATE_COL].max())
print("Rows with unparsed dates:", df[DATE_COL].isna().sum())

flares_per_day = (
    df[df[TARGET] == 1]
    .set_index(DATE_COL)
    .resample("1D")
    .size()
)
plt.figure(figsize=(10, 4))
flares_per_day.plot(kind="line", marker="o", ms=3)
plt.title("M5+ Flare Observations Over Time")
plt.xlabel("Date")
plt.ylabel("Flare rows per day")
plt.tight_layout()
plt.show()

## Target Distribution

In [ ]:
print(df[TARGET].value_counts())

plt.figure(figsize=(6, 4))
sns.countplot(data=df, x=TARGET)
plt.title("Target Distribution")
plt.xlabel("Flare")
plt.ylabel("Count")
plt.show()

## Feature Distributions and Relationship to the Target

Identifier columns (`NOAA_AR`, `HARPNUM`) are excluded from here on — they are not
physical measurements, and plotting or correlating them is not informative (Task 1's
original version included them by mistake).

In [ ]:
FEATURE_COLS = [c for c in df.select_dtypes(include=np.number).columns if c not in ID_COLS + [TARGET]]
print("Feature columns used for EDA:", FEATURE_COLS)

df[FEATURE_COLS].hist(figsize=(16, 12), bins=30)
plt.tight_layout()
plt.show()

In [ ]:
for col in FEATURE_COLS:
    plt.figure(figsize=(6, 4))
    sns.boxplot(data=df, x=TARGET, y=col)
    plt.title(f"{col} vs Flare")
    plt.xlabel("Flare")
    plt.ylabel(col)
    plt.tight_layout()
    plt.show()

## Correlation Analysis

A correlation matrix over the physical features only (IDs and target excluded), with the
strongest pairs printed explicitly rather than only shown visually.

In [ ]:
corr = df[FEATURE_COLS].corr()

plt.figure(figsize=(10, 8))
sns.heatmap(corr, cmap="coolwarm", annot=False)
plt.title("Correlation Heatmap (features only)")
plt.show()

corr_pairs = (
    corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
    .stack()
    .rename("corr")
    .reset_index()
    .rename(columns={"level_0": "feature_1", "level_1": "feature_2"})
)
corr_pairs["abs_corr"] = corr_pairs["corr"].abs()
top_corr = corr_pairs.sort_values("abs_corr", ascending=False).head(10)
display(Markdown("**Top 10 most correlated feature pairs:**"))
display(top_corr[["feature_1", "feature_2", "corr"]])

## Task 1 Summary

(Generated from the actual numbers above — see the imbalance, near-duplicate and
correlation call-outs printed in each cell. Identifier columns `NOAA_AR` and `HARPNUM`
are not physical measurements and are excluded from modeling from Task 2 onward. Because
rows are dense 12-minute snapshots of the same active regions, Task 2 onward splits data
by `NOAA_AR` (or chronologically) rather than randomly, to avoid near-duplicate rows
appearing in both train and test.

# Shared Setup: Grouped Split and Evaluation Helpers

Everything from here on uses a **group-aware split** on `NOAA_AR` so that no active
region's rows appear in both train and test (or across CV folds). Evaluation also adds **PR-AUC**, the
**True Skill Statistic (TSS)** and the **Heidke Skill Score (HSS)**, which are the
standard metrics in the solar-flare-forecasting literature and are far more informative
than accuracy/ROC-AUC at ~3% positive prevalence.

In [ ]:
from sklearn.model_selection import GroupShuffleSplit, StratifiedGroupKFold
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, confusion_matrix,
    ConfusionMatrixDisplay, classification_report, brier_score_loss,
)
from sklearn.utils.class_weight import compute_sample_weight


def tss_hss(y_true, y_pred):
    """True Skill Statistic and Heidke Skill Score from a 2x2 confusion matrix."""
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    tpr = tp / (tp + fn) if (tp + fn) else np.nan   # recall / POD
    fpr = fp / (fp + tn) if (fp + tn) else np.nan
    tss = tpr - fpr
    denom = ((tp + fn) * (fn + tn) + (tp + fp) * (fp + tn))
    hss = 2 * (tp * tn - fp * fn) / denom if denom else np.nan
    return tss, hss


def evaluate(name, y_true, y_pred, y_prob):
    """Compute the full metric set for one model and return it as a dict."""
    tss, hss = tss_hss(y_true, y_pred)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    out = {
        "Model": name,
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred, zero_division=0),
        "Recall": recall_score(y_true, y_pred, zero_division=0),
        "F1": f1_score(y_true, y_pred, zero_division=0),
        "ROC_AUC": roc_auc_score(y_true, y_prob),
        "PR_AUC": average_precision_score(y_true, y_prob),
        "TSS": tss,
        "HSS": hss,
        "TP": tp, "FP": fp, "FN": fn, "TN": tn,
    }
    return out


def narrate(result, context=""):
    """Turn a result dict into a short markdown paragraph, generated live."""
    r = result
    display(Markdown(
        f"**{r['Model']}{(' — ' + context) if context else ''}.** "
        f"Correctly detected {r['TP']} of {r['TP'] + r['FN']} flare events "
        f"(Recall {r['Recall']:.3f}), with {r['FP']} false alarms out of "
        f"{r['FP'] + r['TN']} non-flare rows (Precision {r['Precision']:.3f}). "
        f"F1 = {r['F1']:.3f}, PR-AUC = {r['PR_AUC']:.3f}, "
        f"TSS = {r['TSS']:.3f}, HSS = {r['HSS']:.3f}."
    ))


In [ ]:

groups_all = df["NOAA_AR"]

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=RANDOM_STATE)
train_idx, test_idx = next(gss.split(df, df[TARGET], groups=groups_all))

print(f"Train rows: {len(train_idx)}  (positives: {df.iloc[train_idx][TARGET].sum()})")
print(f"Test rows : {len(test_idx)}  (positives: {df.iloc[test_idx][TARGET].sum()})")

train_regions = set(df.iloc[train_idx]["NOAA_AR"])
test_regions = set(df.iloc[test_idx]["NOAA_AR"])
assert train_regions.isdisjoint(test_regions), "Active regions leaked across the split!"
print("Active regions are disjoint between train and test: OK")


# Task 2 – Handling Class Imbalance

## Objective

The training dataset is highly imbalanced, with significantly more non-flare
observations than flare observations. In this task a baseline Random Forest is trained
without any imbalance-handling technique, then **Class Weighting** and **SMOTE** are
compared. Imputation and oversampling are done inside a
`Pipeline` so nothing is fit on data outside the current training fold, and the
train/test split is the group-aware split defined above, not a random row split.

In [ ]:
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE
from sklearn.ensemble import RandomForestClassifier

FEATURE_COLS_MODEL = [c for c in df.columns if c not in [TARGET, DATE_COL] + ID_COLS]
print("Modeling features (IDs excluded):", FEATURE_COLS_MODEL)

X = df[FEATURE_COLS_MODEL]
y = df[TARGET]

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
groups_train = groups_all.iloc[train_idx]


## Baseline Random Forest

Trained with no imbalance handling, as a reference point. Imputation is fit only on the
training fold inside the pipeline.

In [ ]:
baseline_pipe = ImbPipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("clf", RandomForestClassifier(random_state=RANDOM_STATE)),
])
baseline_pipe.fit(X_train, y_train)

baseline_pred = baseline_pipe.predict(X_test)
baseline_prob = baseline_pipe.predict_proba(X_test)[:, 1]

baseline_result = evaluate("Baseline Random Forest", y_test, baseline_pred, baseline_prob)
narrate(baseline_result, context="no imbalance handling, group-aware split")

ConfusionMatrixDisplay.from_predictions(y_test, baseline_pred)
plt.title("Baseline Random Forest")
plt.show()
print(classification_report(y_test, baseline_pred))

## Strategy 1 – Class Weighting

In [ ]:
weighted_pipe = ImbPipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("clf", RandomForestClassifier(class_weight="balanced", random_state=RANDOM_STATE)),
])
weighted_pipe.fit(X_train, y_train)

weighted_pred = weighted_pipe.predict(X_test)
weighted_prob = weighted_pipe.predict_proba(X_test)[:, 1]

weighted_result = evaluate("Class Weight", y_test, weighted_pred, weighted_prob)
narrate(weighted_result, context="class_weight='balanced'")

ConfusionMatrixDisplay.from_predictions(y_test, weighted_pred)
plt.title("Class-Weighted Random Forest")
plt.show()

## Strategy 2 – SMOTE

SMOTE now lives inside an `imblearn` `Pipeline`, so oversampling happens after the
train/test split and only ever touches the training fold — it is never applied to test
data or fit before splitting.

In [ ]:
smote_pipe = ImbPipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("smote", SMOTE(random_state=RANDOM_STATE)),
    ("clf", RandomForestClassifier(random_state=RANDOM_STATE)),
])
smote_pipe.fit(X_train, y_train)

smote_pred = smote_pipe.predict(X_test)
smote_prob = smote_pipe.predict_proba(X_test)[:, 1]

smote_result = evaluate("SMOTE", y_test, smote_pred, smote_prob)
narrate(smote_result, context="SMOTE inside a pipeline, train fold only")

ConfusionMatrixDisplay.from_predictions(y_test, smote_pred)
plt.title("SMOTE Random Forest")
plt.show()

## Single-Split Comparison

With only ~70 positive test rows, a single train/test split is noisy — a couple of
flipped predictions move Recall by a percentage point or two. The table below is from
this one split; grouped cross-validation below gives a more defensible comparison.

In [ ]:
task2_results = pd.DataFrame([baseline_result, weighted_result, smote_result]).set_index("Model")
task2_results = task2_results[["Accuracy", "Precision", "Recall", "F1", "ROC_AUC", "PR_AUC", "TSS", "HSS"]].round(4)
task2_results

## Grouped Cross-Validation Comparison

To avoid drawing conclusions from one noisy split, each strategy is re-scored with
5-fold `StratifiedGroupKFold` (grouped by `NOAA_AR`, stratified by label), reporting
mean ± standard deviation of F1 and PR-AUC across folds.

In [ ]:
from sklearn.model_selection import cross_validate

cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

cv_pipes = {
    "Baseline Random Forest": ImbPipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("clf", RandomForestClassifier(random_state=RANDOM_STATE)),
    ]),
    "Class Weight": ImbPipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("clf", RandomForestClassifier(class_weight="balanced", random_state=RANDOM_STATE)),
    ]),
    "SMOTE": ImbPipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("smote", SMOTE(random_state=RANDOM_STATE)),
        ("clf", RandomForestClassifier(random_state=RANDOM_STATE)),
    ]),
}

cv_rows = []
for name, pipe in cv_pipes.items():
    scores = cross_validate(
        pipe, X_train, y_train,
        groups=groups_train,
        cv=cv,
        scoring=["f1", "average_precision", "recall"],
        n_jobs=-1,
    )
    cv_rows.append({
        "Model": name,
        "F1 mean": scores["test_f1"].mean(), "F1 std": scores["test_f1"].std(),
        "PR-AUC mean": scores["test_average_precision"].mean(), "PR-AUC std": scores["test_average_precision"].std(),
        "Recall mean": scores["test_recall"].mean(), "Recall std": scores["test_recall"].std(),
    })

task2_cv = pd.DataFrame(cv_rows).set_index("Model").round(4)
task2_cv

In [ ]:
best_cv_strategy = task2_cv.sort_values("F1 mean", ascending=False).index[0]
display(Markdown(
    f"**Task 2 conclusion (generated from the CV table above).** Across grouped "
    f"5-fold CV, **{best_cv_strategy}** had the highest mean F1 "
    f"({task2_cv.loc[best_cv_strategy, 'F1 mean']:.3f} ± {task2_cv.loc[best_cv_strategy, 'F1 std']:.3f}). "
    f"With this few positive examples, treat differences smaller than roughly one "
    f"standard deviation as noise rather than a real ranking. This strategy "
    f"({best_cv_strategy}) is carried forward as the imbalance-handling choice for "
    f"Task 3 and Task 4."
))
BEST_IMBALANCE_STRATEGY = best_cv_strategy

# Task 3 – Feature Engineering

## Objective

Five physically motivated features are derived from the magnetic field measurements.
Two bugs from the original version are fixed here:

1. `df.replace(-1, np.nan)` was applied to the **whole dataframe**, but `R_VALUE`'s
   legitimate minimum is `-1.0` — that line was silently deleting real data on every
   column, not just a sentinel value. It is removed; there is no evidence `-1` is a
   missing-value sentinel anywhere in this file.
2. `HELICITY_RATIO = TOTUSJH / ABSNJZH` divides by a value that can be at or near zero,
   which produces `inf`/huge values rather than a clean `NaN`. A safe divide (returning
   `NaN` when the denominator is ~0) is used instead for every ratio feature.

Ratios of already-normalized values (the raw columns are already scaled to roughly
`[-1, 0]`) don't carry the same physical meaning as ratios of raw physical units — this
is flagged, not silently assumed to be fine, and the ablation below checks empirically
whether the engineered features actually help.

In [ ]:
def safe_divide(numerator, denominator, eps=1e-6):
    denom = denominator.where(denominator.abs() > eps, np.nan)
    return numerator / denom


def add_engineered_features(frame):
    out = frame.copy()
    out["ENERGY_DENSITY"] = safe_divide(out["TOTPOT"], out["AREA_ACR"])
    out["FLUX_DENSITY"] = safe_divide(out["USFLUX"], out["AREA_ACR"])
    out["CURRENT_DENSITY"] = safe_divide(out["TOTUSJZ"], out["AREA_ACR"])
    out["TWIST_ENERGY"] = out["MEANALP"] * np.sign(out["TOTPOT"]) * np.log1p(out["TOTPOT"].abs())
    out["HELICITY_RATIO"] = safe_divide(out["TOTUSJH"], out["ABSNJZH"])
    return out


ENGINEERED_COLS = ["ENERGY_DENSITY", "FLUX_DENSITY", "CURRENT_DENSITY", "TWIST_ENERGY", "HELICITY_RATIO"]

df_fe = add_engineered_features(df)

inf_counts = np.isinf(df_fe[ENGINEERED_COLS]).sum()
nan_counts = df_fe[ENGINEERED_COLS].isna().sum()
print("Inf counts per engineered feature:\n", inf_counts)
print("\nNaN counts per engineered feature:\n", nan_counts)

df_fe[ENGINEERED_COLS] = df_fe[ENGINEERED_COLS].replace([np.inf, -np.inf], np.nan)


## Ablation: With vs. Without Engineered Features

Instead of comparing against hard-coded numbers from Task 2, both variants are computed from the same grouped CV machinery, on the same folds,
so the comparison is apples-to-apples.

In [ ]:
X_base = df_fe[FEATURE_COLS_MODEL]
X_with_eng = df_fe[FEATURE_COLS_MODEL + ENGINEERED_COLS]
y_fe = df_fe[TARGET]

def make_pipe(strategy_name):
    steps = [("imputer", SimpleImputer(strategy="median"))]
    if strategy_name == "SMOTE":
        steps.append(("smote", SMOTE(random_state=RANDOM_STATE)))
    clf_kwargs = {"random_state": RANDOM_STATE}
    if strategy_name in ("Class Weight", "Baseline Random Forest"):
        if strategy_name == "Class Weight":
            clf_kwargs["class_weight"] = "balanced"
    steps.append(("clf", RandomForestClassifier(**clf_kwargs)))
    return ImbPipeline(steps)


ablation_rows = []
for feat_name, X_variant in [("Without engineered features", X_base), ("With engineered features", X_with_eng)]:
    X_tr, X_te = X_variant.iloc[train_idx], X_variant.iloc[test_idx]
    y_tr, y_te = y_fe.iloc[train_idx], y_fe.iloc[test_idx]

    pipe = make_pipe(BEST_IMBALANCE_STRATEGY)
    scores = cross_validate(
        pipe, X_tr, y_tr, groups=groups_train, cv=cv,
        scoring=["f1", "average_precision", "recall"], n_jobs=-1,
    )
    ablation_rows.append({
        "Feature set": feat_name,
        "F1 mean": scores["test_f1"].mean(), "F1 std": scores["test_f1"].std(),
        "PR-AUC mean": scores["test_average_precision"].mean(),
        "Recall mean": scores["test_recall"].mean(),
    })

ablation_df = pd.DataFrame(ablation_rows).set_index("Feature set").round(4)
ablation_df

In [ ]:
eng_helps = ablation_df.loc["With engineered features", "F1 mean"] > ablation_df.loc["Without engineered features", "F1 mean"]
delta = ablation_df.loc["With engineered features", "F1 mean"] - ablation_df.loc["Without engineered features", "F1 mean"]

display(Markdown(
    f"**Task 3 conclusion (from the ablation above).** The engineered features changed "
    f"mean grouped-CV F1 by {delta:+.4f} "
    f"({'an improvement' if eng_helps else 'no improvement, or a regression'}). "
    f"{'They are kept for Task 4.' if eng_helps else 'Given this, Task 4 proceeds without them unless the margin is judged worth the added complexity.'}"
))
USE_ENGINEERED_FEATURES = bool(eng_helps)

FINAL_FEATURE_SET = (FEATURE_COLS_MODEL + ENGINEERED_COLS) if USE_ENGINEERED_FEATURES else FEATURE_COLS_MODEL
X_final = df_fe[FINAL_FEATURE_SET]
y_final = df_fe[TARGET]

X_train_fe, X_test_fe = X_final.iloc[train_idx], X_final.iloc[test_idx]
y_train_fe, y_test_fe = y_final.iloc[train_idx], y_final.iloc[test_idx]

## Feature Importance

Fit once on the full training fold (with the imputer inside a pipeline) to see which
features — original or engineered — the model actually relies on.

In [ ]:
importance_pipe = ImbPipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("clf", RandomForestClassifier(class_weight="balanced", random_state=RANDOM_STATE)),
])
importance_pipe.fit(X_train_fe, y_train_fe)

importances = pd.Series(
    importance_pipe.named_steps["clf"].feature_importances_,
    index=X_train_fe.columns,
).sort_values(ascending=False)

plt.figure(figsize=(8, 6))
importances.plot(kind="barh")
plt.gca().invert_yaxis()
plt.title("Random Forest Feature Importances")
plt.tight_layout()
plt.show()

importances

# Task 4 – Model Training and Comparison

## Objective

Five tree-based algorithms are compared: Decision Tree, Random Forest, AdaBoost,
Gradient Boosting and XGBoost.

**The final model is chosen by cross-validated training performance, not by looking
   at the test set.** The test set is touched exactly once, after selection, to report
   the final unbiased estimate.

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import AdaBoostClassifier, GradientBoostingClassifier
from sklearn.model_selection import RandomizedSearchCV
from xgboost import XGBClassifier

sample_weight_train = compute_sample_weight("balanced", y_train_fe)

imputer_task4 = SimpleImputer(strategy="median")
X_train_imp = pd.DataFrame(
    imputer_task4.fit_transform(X_train_fe), columns=X_train_fe.columns, index=X_train_fe.index
)
X_test_imp = pd.DataFrame(
    imputer_task4.transform(X_test_fe), columns=X_test_fe.columns, index=X_test_fe.index
)
print("Missing values in X_train_imp:", X_train_imp.isnull().sum().sum())
print("Missing values in X_test_imp :", X_test_imp.isnull().sum().sum())

## Hyperparameter Search

Modest search spaces are used (the positive class has only a few dozen examples in any
given training fold, so an aggressive search would mostly tune noise). Each model is
scored on PR-AUC via `StratifiedGroupKFold`, grouped by `NOAA_AR`.

In [ ]:
N_ITER_SEARCH = 8
cv_fast = StratifiedGroupKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)

param_distributions = {
    "Decision Tree": (
        DecisionTreeClassifier(random_state=RANDOM_STATE),
        {
            "max_depth": [4, 6, 8, 10, None],
            "min_samples_split": [2, 5, 10, 20],
            "min_samples_leaf": [1, 2, 5, 10],
        },
    ),
    "Random Forest": (
        RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1),
        {
            "n_estimators": [100, 200],
            "max_depth": [6, 10, 12, None],
            "min_samples_split": [2, 5, 10],
            "min_samples_leaf": [1, 2, 5],
        },
    ),
    "AdaBoost": (
        AdaBoostClassifier(random_state=RANDOM_STATE),
        {
            "n_estimators": [50, 100, 150],
            "learning_rate": [0.05, 0.1, 0.5, 0.8, 1.0],
        },
    ),
    "Gradient Boosting": (
        GradientBoostingClassifier(random_state=RANDOM_STATE),
        {
            "n_estimators": [100, 150],
            "learning_rate": [0.03, 0.05, 0.08, 0.1],
            "max_depth": [2, 3, 4],
            "subsample": [0.7, 0.8, 1.0],
        },
    ),
    "XGBoost": (
        XGBClassifier(random_state=RANDOM_STATE, eval_metric="logloss", n_jobs=-1),
        {
            "n_estimators": [100, 150, 200],
            "max_depth": [3, 4, 5],
            "learning_rate": [0.05, 0.08, 0.1],
            "subsample": [0.7, 0.8, 1.0],
            "colsample_bytree": [0.7, 0.8, 1.0],
        },
    ),
}

tuned_models = {}
for name, (estimator, grid) in param_distributions.items():
    search = RandomizedSearchCV(
        estimator,
        param_distributions=grid,
        n_iter=N_ITER_SEARCH,
        scoring="average_precision",
        cv=cv_fast,
        n_jobs=-1,
        random_state=RANDOM_STATE,
    )
    # RandomizedSearchCV's final refit (refit=True, the default) reuses sample_weight
    # from this call, so tuned_models[name] below is already fit on the full training
    # fold with weights applied — no need to refit it again later.
    search.fit(X_train_imp, y_train_fe, groups=groups_train, sample_weight=sample_weight_train)
    tuned_models[name] = search.best_estimator_
    print(f"{name}: best PR-AUC(CV) = {search.best_score_:.4f}, best params = {search.best_params_}")

## Cross-Validated Comparison (used for model selection)

Each already-tuned model is re-scored with `cross_validate` on the training data only
(same fast 3-fold grouped CV, with `sample_weight` passed through), to get a selection
metric that never touches the test set.

In [ ]:
cv_select_rows = []
for name, model in tuned_models.items():
    scores = cross_validate(
        model, X_train_imp, y_train_fe,
        groups=groups_train, cv=cv_fast,
        scoring=["f1", "average_precision", "recall"],
        params={"sample_weight": sample_weight_train},
        n_jobs=-1,
    )
    cv_select_rows.append({
        "Model": name,
        "F1 mean": scores["test_f1"].mean(), "F1 std": scores["test_f1"].std(),
        "PR-AUC mean": scores["test_average_precision"].mean(), "PR-AUC std": scores["test_average_precision"].std(),
        "Recall mean": scores["test_recall"].mean(),
    })

task4_cv = pd.DataFrame(cv_select_rows).set_index("Model").round(4)
task4_cv.sort_values("F1 mean", ascending=False)

In [ ]:
FINAL_MODEL_NAME = task4_cv.sort_values("F1 mean", ascending=False).index[0]
display(Markdown(
    f"**Model selected for final evaluation: {FINAL_MODEL_NAME}**, chosen by mean "
    f"grouped-CV F1 on the training data only "
    f"({task4_cv.loc[FINAL_MODEL_NAME, 'F1 mean']:.3f} ± {task4_cv.loc[FINAL_MODEL_NAME, 'F1 std']:.3f}). "
    f"With only a few dozen positive examples per fold, treat any ranking narrower than "
    f"roughly one standard deviation as a tie, not a real winner."
))

## Final, Single Evaluation on the Held-Out Test Set

The selected model is fit on the full training fold and evaluated on the test fold
**once** — this is the only place the test set is used for anything other than
inspection.

In [ ]:

final_model = tuned_models[FINAL_MODEL_NAME]

final_pred = final_model.predict(X_test_imp)
final_prob = final_model.predict_proba(X_test_imp)[:, 1]

final_result = evaluate(FINAL_MODEL_NAME, y_test_fe, final_pred, final_prob)
narrate(final_result, context="final held-out test evaluation")

ConfusionMatrixDisplay.from_predictions(y_test_fe, final_pred, cmap="Blues")
plt.title(f"Final Selected Model — {FINAL_MODEL_NAME}")
plt.show()

print(classification_report(y_test_fe, final_pred))

## All Five Models on the Test Set (for reference only)

Shown for comparison, but **not** used to pick the final model — selection already
happened above using CV on the training data.

In [ ]:
# All five are already fit (via RandomizedSearchCV's refit) -- just predict, no refitting.
test_rows = []
for name, model in tuned_models.items():
    pred = model.predict(X_test_imp)
    prob = model.predict_proba(X_test_imp)[:, 1]
    test_rows.append(evaluate(name, y_test_fe, pred, prob))

task4_test = pd.DataFrame(test_rows).set_index("Model")
task4_test = task4_test[["Accuracy", "Precision", "Recall", "F1", "ROC_AUC", "PR_AUC", "TSS", "HSS"]].round(4)
task4_test

In [ ]:
from sklearn.metrics import roc_curve

plt.figure(figsize=(8, 6))
for name, model in tuned_models.items():
    prob = model.predict_proba(X_test_imp)[:, 1]
    fpr, tpr, _ = roc_curve(y_test_fe, prob)
    plt.plot(fpr, tpr, label=f"{name} (AUC={roc_auc_score(y_test_fe, prob):.4f})")
plt.plot([0, 1], [0, 1], "k--")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve Comparison (test set, reference only)")
plt.legend(loc="lower right")
plt.show()

## Threshold Tuning and Calibration

The default 0.5 decision threshold is rarely optimal for a ~3% positive class. The
precision-recall curve is used to find a threshold that favors Recall appropriately for
an early-warning use case, and the reliability curve / Brier score check how trustworthy
the predicted probabilities themselves are.

In [ ]:
from sklearn.metrics import precision_recall_curve
from sklearn.calibration import calibration_curve

precisions, recalls, thresholds = precision_recall_curve(y_test_fe, final_prob)
f1_by_threshold = 2 * precisions * recalls / (precisions + recalls + 1e-12)
best_thr_idx = np.nanargmax(f1_by_threshold[:-1])
best_threshold = thresholds[best_thr_idx]

display(Markdown(
    f"**Best F1-maximizing threshold on the test set:** {best_threshold:.3f} "
    f"(F1 = {f1_by_threshold[best_thr_idx]:.3f} vs {final_result['F1']:.3f} at the "
    f"default 0.5 threshold). This is reported for context, not re-optimized on test — "
    f"in a real deployment this threshold should be chosen on a validation fold, not test."
))

plt.figure(figsize=(6, 5))
plt.plot(recalls, precisions)
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title(f"Precision-Recall Curve — {FINAL_MODEL_NAME}")
plt.show()

frac_pos, mean_pred = calibration_curve(y_test_fe, final_prob, n_bins=10, strategy="quantile")
plt.figure(figsize=(6, 5))
plt.plot(mean_pred, frac_pos, marker="o", label=FINAL_MODEL_NAME)
plt.plot([0, 1], [0, 1], "k--", label="Perfectly calibrated")
plt.xlabel("Mean predicted probability")
plt.ylabel("Observed frequency")
plt.title("Reliability Curve")
plt.legend()
plt.show()

print("Brier score:", brier_score_loss(y_test_fe, final_prob))

## Task 4 Conclusion


In [ ]:
cv_ranked = task4_cv.sort_values("F1 mean", ascending=False)
top_model, runner_up = cv_ranked.index[0], cv_ranked.index[1]
gap = cv_ranked.loc[top_model, "F1 mean"] - cv_ranked.loc[runner_up, "F1 mean"]
is_tie = gap < cv_ranked.loc[top_model, "F1 std"]

display(Markdown(
    f"## Task 4 Conclusion\n\n"
    f"Five tree-based models were tuned with `RandomizedSearchCV` under grouped CV and "
    f"compared on cross-validated F1 computed only on training data — the test set was "
    f"never used for selection. **{FINAL_MODEL_NAME}** was selected on that basis "
    f"(CV F1 {cv_ranked.loc[top_model, 'F1 mean']:.3f} ± {cv_ranked.loc[top_model, 'F1 std']:.3f}), "
    f"although it is " + ("statistically tied" if is_tie else "clearly ahead of") +
    f" {runner_up} (CV F1 {cv_ranked.loc[runner_up, 'F1 mean']:.3f} ± {cv_ranked.loc[runner_up, 'F1 std']:.3f}); "
    f"with this few positive examples per fold, a gap of {gap:.3f} is within noise and "
    f"the two should be considered comparable rather than a clean win.\n\n"
    f"On the held-out test set — touched once, after selection — {FINAL_MODEL_NAME} detected "
    f"{final_result['TP']} of {final_result['TP'] + final_result['FN']} flare events "
    f"(**Recall {final_result['Recall']:.3f}**) with {final_result['FP']} false alarms "
    f"(Precision {final_result['Precision']:.3f}), F1 {final_result['F1']:.3f}, "
    f"PR-AUC {final_result['PR_AUC']:.3f}, **TSS {final_result['TSS']:.3f}**, "
    f"HSS {final_result['HSS']:.3f}. A TSS in this range is consistent with published "
    f"M-class flare forecasting work, and the low precision is a deliberate trade-off: "
    f"in this problem a missed flare is far costlier than a false alarm.\n\n"
    f"**Limitations.** (1) {FINAL_MODEL_NAME}'s predicted probabilities are poorly calibrated "
    f"(see the reliability curve above) — a known property of boosted classifiers under "
    f"this loss, so its probability outputs should not be trusted directly without "
    f"recalibration (e.g. `CalibratedClassifierCV`). (2) The test-set numbers above come "
    f"from one split; the grouped-CV numbers used for selection are the more defensible "
    f"estimate of generalization. (3) Given the CV tie, Gradient Boosting is a reasonable "
    f"alternative, particularly if calibrated probabilities matter more than raw recall."
))